# Validate Fabric-Care Measurements Before Calculating | Sewlore

A CSV is a small table stored as text. It can keep sample notes together, but a table is not evidence that its entries are usable. This lesson checks your records before calculating signed length and width changes. It begins with a header and no measurements. Nothing in the blank input represents a fabric test. For a single pair, the [Fabric Shrinkage Calculator](https://sewlore-fabric-shrinkage.web.app/) offers the same kind of arithmetic in a browser form.

## Record a pair that can be compared

Use the same marked spans on the same swatch before and after the care method actually used. Identify length and width, retain the sample's starting condition, and measure in comparable conditions. Keep a project or pattern revision in your notes when relevant. The [before-cutting record guide](https://sewlore-preparation-notes.blogspot.com/2026/10/what-to-record-before-cutting-fabric.html) explains these connections. Follow appropriate care guidance from the fabric manufacturer; the code does not prescribe a washing or drying treatment.

## Keep the supplied columns

`sample_id` identifies the record's material. `before_unit` and `after_unit` must both be `cm`, `mm` or `in`, and must match. The four length/width fields need finite decimal numbers greater than zero. Put units in the unit columns rather than adding “cm” to a number. `care_method` is required context; `notes` can preserve condition, dates, settings and observations. Commas, quotes and line breaks inside a field need normal CSV quoting. Using a spreadsheet's CSV export usually handles that structure.

Set `rectangle_assumption` to `yes` only when both sets of sides describe rectangles. Use `no` or leave it blank otherwise. Keeping the header in its supplied order makes this small lesson predictable. Missing or additional cells are reported instead of guessed. A completely empty line is skipped; a partly completed record needs correction.

## Validate first, then interpret

Run the definitions cell, add your own rows to `RAW_CSV`, and run validation. Accepted records and diagnostics are separate. Correct an error in the original CSV and rerun; do not copy a rejected row into the calculated output. Record numbers count CSV records with the header as record 1. Quoted notes containing a line break remain one record.

The axis formula is `(before − after) / before × 100`. Positive means contraction; negative means expansion. The code retains the sign. Units normalize internally using 1 inch = 25.4 mm and 1 cm = 10 mm. Extremely large or small numeric combinations that cannot be represented reliably are rejected. Numerical precision does not certify the accuracy of your measuring tool.

The optional area result compares products of perpendicular sides. Axis percentages do not simply add together, and a skewed grid or irregular shape needs another area measurement. Keep care readings separate from pull-and-release observations; the [stretch recording method](https://sewlore.com/blogs/sewing-journal/measure-fabric-stretch-and-record-recovery) addresses that different check. Review actual material requirements with the [fabric and elastic preparation guide](https://sewlore.com/blogs/sewing-journal/choose-fabric-and-elastic-for-sewing). None of these calculations establishes garment fit or laboratory performance.

## Export a record, protect the context

The export cell builds two CSV strings: accepted records and errors. Optional saving is off initially. Enable it only when you want files in your current runtime, then download them through Colab's Files panel. Formula-like free text receives a leading apostrophe in exported cells; real numeric negatives remain numbers. This changes the export text, not your internal notes. Spreadsheet import behaviour varies, so inspect the exported text before using it.

Colab processes text in its account runtime. This notebook adds no network requests, Drive mounting, account access or analytics. Shared notebooks include saved code and outputs: publish a blank copy and clear outputs first. Keep personal measurements and identifying notes private. If you also use a printed pattern, follow its [printing and assembly guidance](https://sewlore.com/pages/printing-assembly); a CSV is not a print-scale reference.

Optional software assertions use explicitly hypothetical numbers to check the code. They are not a measured dataset or physical care results.


## 1. Run the definitions

Standard-library code; no installs or network calls.


In [ ]:
"""Original Sewlore lesson code. No network, account access or physical test data."""
import csv
import io
import math
import re

INPUT_COLUMNS = (
    "sample_id", "before_unit", "after_unit", "before_length", "before_width",
    "after_length", "after_width", "care_method", "rectangle_assumption", "notes",
)
OUTPUT_COLUMNS = INPUT_COLUMNS + (
    "length_change_percent", "width_change_percent", "rectangular_area_change_percent",
)
ERROR_COLUMNS = ("record_number", "sample_id", "field", "message")
UNIT_MM = {"mm": 1.0, "cm": 10.0, "in": 25.4}
DECIMAL = re.compile(r"^\+?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:e[+-]?[0-9]+)?$", re.I)
BLANK_CSV = ",".join(INPUT_COLUMNS) + "\n"

def positive_number(value):
    if isinstance(value, bool) or not isinstance(value, (str, int, float)):
        raise ValueError("Use a finite decimal number greater than zero.")
    if isinstance(value, str) and not DECIMAL.fullmatch(value.strip()):
        raise ValueError("Use a decimal number with no unit text or thousands separator.")
    number = float(value)
    if not math.isfinite(number) or number <= 0:
        raise ValueError("The reading must be finite and greater than zero.")
    return number

def to_mm(value, unit):
    if unit not in UNIT_MM:
        raise ValueError("The unit must be cm, mm or in.")
    result = positive_number(value) * UNIT_MM[unit]
    if not math.isfinite(result) or result <= 0:
        raise ValueError("The converted length is outside the supported numeric range.")
    return result

def convert_length(value, from_unit, to_unit):
    if to_unit not in UNIT_MM:
        raise ValueError("The destination unit must be cm, mm or in.")
    result = to_mm(value, from_unit) / UNIT_MM[to_unit]
    if not math.isfinite(result) or result <= 0:
        raise ValueError("The conversion is outside the supported numeric range.")
    return result

def calculate_changes(before_length_mm, before_width_mm, after_length_mm, after_width_mm, rectangle_assumption=False):
    if not isinstance(rectangle_assumption, bool):
        raise ValueError("The rectangle assumption must be explicitly True or False.")
    before_length_mm, before_width_mm, after_length_mm, after_width_mm = map(
        positive_number, (before_length_mm, before_width_mm, after_length_mm, after_width_mm)
    )
    length_ratio = after_length_mm / before_length_mm
    width_ratio = after_width_mm / before_width_mm
    if not all(math.isfinite(r) and r > 0 for r in (length_ratio, width_ratio)):
        raise ValueError("These magnitudes cannot be compared reliably.")
    length = (1 - length_ratio) * 100
    width = (1 - width_ratio) * 100
    if not all(math.isfinite(r) for r in (length, width)):
        raise ValueError("The axis change is outside the supported numeric range.")
    area = None
    if rectangle_assumption:
        ratio = length_ratio * width_ratio
        area = (1 - ratio) * 100
        if not math.isfinite(area) or ratio <= 0:
            raise ValueError("The requested rectangular area change cannot be represented reliably.")
    return {"length_change_percent": length, "width_change_percent": width,
            "rectangular_area_change_percent": area}

def validate_csv(raw_csv):
    """Return separately validated records and diagnostics. Header is logical record 1."""
    if not isinstance(raw_csv, str):
        raise TypeError("Provide CSV as text.")
    if len(raw_csv) > 1_000_000:
        return [], [{"record_number": 0, "sample_id": "", "field": "CSV", "message": "Use a CSV smaller than one million characters."}]
    diagnostics = []
    def issue(number, sample_id, field, message):
        diagnostics.append({"record_number": number, "sample_id": sample_id, "field": field, "message": message})
    try:
        reader = csv.reader(io.StringIO(raw_csv.removeprefix("\ufeff"), newline=""), strict=True)
        header = next(reader, None)
        if header != list(INPUT_COLUMNS):
            issue(1, "", "header", "Use the exact supplied header, once, in its supplied order.")
            return [], diagnostics
        rows = list(reader)  # Parse the entire structure before calculating any record.
    except csv.Error as error:
        issue(0, "", "CSV", f"CSV structure could not be read: {error}")
        return [], diagnostics
    validated = []
    for number, cells in enumerate(rows, start=2):
        if not cells:  # Ignore a completely empty line, never a partly empty record.
            continue
        sample_id = cells[0].strip() if cells else ""
        if len(cells) != len(INPUT_COLUMNS):
            issue(number, sample_id, "columns", f"Expected {len(INPUT_COLUMNS)} cells; found {len(cells)}.")
            continue
        row = dict(zip(INPUT_COLUMNS, cells))
        start_errors = len(diagnostics)
        row["sample_id"] = sample_id
        row["care_method"] = row["care_method"].strip()
        for field in ("sample_id", "care_method"):
            if not row[field]: issue(number, sample_id, field, "Record this value before comparing the sample.")
        for field in ("before_unit", "after_unit"):
            row[field] = row[field].strip().lower()
            if row[field] not in UNIT_MM: issue(number, sample_id, field, "Use cm, mm or in.")
        if row["before_unit"] in UNIT_MM and row["after_unit"] in UNIT_MM and row["before_unit"] != row["after_unit"]:
            issue(number, sample_id, "units", "Before and after units must match. Convert the readings first.")
        assumption = row["rectangle_assumption"].strip().lower()
        if assumption not in ("", "yes", "no"):
            issue(number, sample_id, "rectangle_assumption", "Use yes only for actual rectangles; use no or leave blank otherwise.")
        row["rectangle_assumption"] = "yes" if assumption == "yes" else "no"
        mm = {}
        for field in ("before_length", "before_width", "after_length", "after_width"):
            unit = row["before_unit"] if field.startswith("before_") else row["after_unit"]
            try:
                row[field] = positive_number(row[field])
                mm[field] = to_mm(row[field], unit)
            except (ValueError, OverflowError) as error:
                issue(number, sample_id, field, str(error))
        if len(diagnostics) != start_errors:
            continue
        try:
            changes = calculate_changes(mm["before_length"], mm["before_width"], mm["after_length"], mm["after_width"], assumption == "yes")
            validated.append({**row, **changes})
        except ValueError as error:
            issue(number, sample_id, "calculation", str(error))
    return validated, diagnostics

def spreadsheet_safe(value):
    """Neutralize formula-like free text in exports; do not alter internal notes."""
    if value is None:
        return ""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if not math.isfinite(value): raise ValueError("Cannot export a non-finite numeric value.")
        return format(value, ".12g")  # Genuine numeric negatives remain numbers.
    text = str(value)
    if text.startswith(("\t", "\r", "\n")) or text.lstrip().startswith(("=", "+", "-", "@")):
        return "'" + text
    return text

def export_csv(records, columns):
    buffer = io.StringIO(newline="")
    writer = csv.writer(buffer, quoting=csv.QUOTE_ALL, lineterminator="\n")
    writer.writerow(columns)
    for row in records:
        writer.writerow([spreadsheet_safe(row.get(field)) for field in columns])
    return buffer.getvalue()

def export_validated(records):
    return export_csv(records, OUTPUT_COLUMNS)

def export_errors(diagnostics):
    return export_csv(diagnostics, ERROR_COLUMNS)


## 2. Add your records

Keep this public copy header-only. Add your own CSV rows only in your private working copy.


In [ ]:
RAW_CSV = 'sample_id,before_unit,after_unit,before_length,before_width,after_length,after_width,care_method,rectangle_assumption,notes\n'


## 3. Validate before calculating


In [ ]:
validated_records, validation_errors = validate_csv(RAW_CSV)
print(f"Validated records: {len(validated_records)}; diagnostics: {len(validation_errors)}")
for diagnostic in validation_errors:
    print(diagnostic)


## 4. Build both CSV exports


In [ ]:
VALIDATED_CSV = export_validated(validated_records)
ERRORS_CSV = export_errors(validation_errors)
print("Accepted-record and error CSV buffers are ready. They contain only headers when input has no records.")


## 5. Optional: save runtime files

Set the switch to True to save, then use the Files panel to download. No Drive access is used.


In [ ]:
SAVE_EXPORT_FILES = False
if SAVE_EXPORT_FILES:
    from pathlib import Path
    Path("validated-fabric-care.csv").write_text(VALIDATED_CSV, encoding="utf-8")
    Path("fabric-care-validation-errors.csv").write_text(ERRORS_CSV, encoding="utf-8")
    print("Saved the two CSV files in this runtime. Inspect and download them before the runtime ends.")
else:
    print("Saving is off. CSV buffers remain in memory.")


## 6. Optional software checks — hypothetical fixtures

The following numbers are synthetic software assertions, not physical observations. They never populate RAW_CSV.


In [ ]:
def run_hypothetical_software_checks():
    # Hypothetical numeric fixture: no swatch was measured or washed.
    result = calculate_changes(200, 200, 190, 195, True)
    assert math.isclose(result["length_change_percent"], 5, abs_tol=1e-10)
    assert math.isclose(result["width_change_percent"], 2.5, abs_tol=1e-10)
    assert math.isclose(result["rectangular_area_change_percent"], 7.375, abs_tol=1e-10)
    expansion = calculate_changes(200, 200, 210, 220, True)
    assert math.isclose(expansion["length_change_percent"], -5, abs_tol=1e-10)
    assert math.isclose(expansion["rectangular_area_change_percent"], -15.5, abs_tol=1e-10)
    assert convert_length(.625, "in", "mm") == 15.875
    assert calculate_changes(20, 20, 19, 19.5, False)["rectangular_area_change_percent"] is None
    assert validate_csv(BLANK_CSV) == ([], [])
    assert spreadsheet_safe("=synthetic-test") == "'=synthetic-test"
    print("Hypothetical software assertions passed; no fabric dataset or care result was created.")

run_hypothetical_software_checks()


## Original code and lesson licence

MIT License

Copyright (c) 2026 Sewlore

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

Sewlore names and branding remain the property of their respective owner. This
licence does not grant trademark ownership or imply endorsement.
